# A/B Testing Kit — walkthrough

Every number in this notebook is generated by running the code, against simulated experiments
whose true effect is **known**. That is the whole point of the repository: a claim about an
error rate is only checkable if you know what the right answer was.

The notebook is a demonstration layer. The headline figures in the README come from long CLI
runs at 20,000 replications; here the replication counts are cut so the whole thing executes
in about a minute, which means the intervals are wider and the numbers wobble in the third
decimal. The *shapes* are identical.

**Scenario throughout:** a checkout flow converting at 10%, and we want to detect a 2
percentage-point absolute lift (a 20% relative improvement) at α = 0.05, two-sided, 80% power.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ab_testing_kit.bayes import DecisionRule, agreement_study, bayes_peeking_study
from ab_testing_kit.cuped import variance_reduction_curve
from ab_testing_kit.design import analytic_power, mde, sample_size, validate_power
from ab_testing_kit.peeking import type_i_error_curve
from ab_testing_kit.sequential import sequential_design, sequential_error_curve, sequential_study
from ab_testing_kit.simulation import ExperimentSpec

# Small enough to execute quickly, large enough for the pattern to be unambiguous.
N_SIMS = 400
LOOK_COUNTS = (1, 2, 3, 5, 10, 20)

BASELINE, EFFECT = 0.10, 0.02
plt.rcParams.update({"figure.figsize": (9, 5), "axes.grid": True, "grid.alpha": 0.3})
NAVY, RUST, TEAL, PLUM = "#1f3a5f", "#c1440e", "#2a9d8f", "#7b506f"

## 1. Design: how many users do we need?

The sample-size formula is sized for the *pooled* two-proportion z-test, which is the test
`fixed_horizon_test` actually applies. Sizing for one test and analysing with another is a
quiet route to being under-powered, so the two are deliberately kept in step — and then
checked against simulation rather than trusted.

In [ ]:
n_required = sample_size("binary", BASELINE, EFFECT)
print(f"n per arm      {n_required:,}")
print(f"total n        {2 * n_required:,}")
print(f"analytic power {analytic_power('binary', BASELINE, EFFECT, n_required):.4f}")

check = validate_power(
    ExperimentSpec(n_per_arm=n_required, metric="binary", baseline=BASELINE, effect=EFFECT),
    n_sims=N_SIMS * 3,
    seed=0,
)
lo, hi = check.simulated.reject_rate_ci
print(f"\nsimulated power {check.simulated.reject_rate:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"formula inside the interval: {check.agrees}")

The formula lands inside the Monte Carlo interval. That is the baseline everything else is
measured against: analysed **once**, at the planned horizon, the test does what it says.

### What could we detect with the traffic we already have?

In [ ]:
traffic = np.array([500, 1000, 2000, 5000, 10000, 20000])
detectable = np.array([mde("binary", BASELINE, int(n)) for n in traffic])

fig, ax = plt.subplots()
ax.plot(traffic, detectable * 100, "o-", color=NAVY, lw=2)
ax.axhline(EFFECT * 100, color=RUST, ls="--", label=f"target lift = {EFFECT * 100:.0f} pp")
ax.axvline(n_required, color=TEAL, ls=":", label=f"required n = {n_required:,}")
ax.set_xscale("log")
ax.set_xlabel("users per arm")
ax.set_ylabel("minimum detectable effect (percentage points)")
ax.set_title("What a given amount of traffic can detect (10% baseline, 80% power)")
ax.legend()
plt.show()

## 2. Peeking: what checking the dashboard actually costs

Now the same data, analysed repeatedly as it accumulates, stopping the first time it goes
green. Nothing else changes — same generated experiments, same seed, same nominal α.

Two implementation details make this an honest model rather than a strawman. Units are stored
in **arrival order**, so `truncate(k)` really is "the data as it looked on day 3"; and
consecutive looks therefore *share data*, which is what makes them correlated. Regenerating a
smaller experiment at each look would make the looks independent and badly understate the
damage.

In [ ]:
null_spec = ExperimentSpec(n_per_arm=n_required, metric="binary", baseline=BASELINE)

naive = type_i_error_curve(null_spec, look_counts=LOOK_COUNTS, n_sims=N_SIMS, seed=0)
for k, summary in naive.items():
    lo, hi = summary.reject_rate_ci
    print(f"{k:>3} looks   Type-I {summary.reject_rate:.4f}   95% CI [{lo:.4f}, {hi:.4f}]")

## 3. Alpha spending: keep the looks, budget the error

The fix is not "look less often". It is to decide in advance how much of the 5% each look may
consume, and raise the bar accordingly so the total still comes to 5%.

The boundary below is **computed**, not quoted from a table — there is no closed form past the
first look, because look 3 contains look 2's data. It comes from a recursive integration over
the Brownian sub-density of the score statistic, validated in the test suite against a direct
simulation of the process it claims to integrate.

In [ ]:
fig, ax = plt.subplots()
for spending, colour in (("obrien-fleming", NAVY), ("pocock", PLUM)):
    design = sequential_design(n_required, 10, spending=spending)
    finite = [(t, z) for t, z in zip(design.boundary.information, design.boundary.z) if np.isfinite(z)]
    ax.plot(*zip(*finite), "o-", color=colour, lw=2, label=spending)

ax.axhline(1.959964, color=RUST, ls="--", label="uncorrected 1.96")
ax.set_xlabel("information fraction (n at this look / planned n)")
ax.set_ylabel("critical |z| required to stop")
ax.set_title("The significance bar, look by look")
ax.legend()
plt.show()

design = sequential_design(n_required, 5)
print("Five-look O'Brien-Fleming boundary:")
for i, (n, z, spent) in enumerate(
    zip(design.schedule, design.boundary.z, design.boundary.achieved), start=1
):
    print(f"  look {i}  n={n:>6,}  z={z:>7.4f}  cumulative alpha spent {spent:.6f}")
print(f"\ntotal alpha spent {design.boundary.total_alpha:.6f}")

O'Brien–Fleming refuses to stop early unless the result is overwhelming — the first of five
looks demands z = 4.38 — which is exactly why its final look sits at 2.06, barely above the
uncorrected 1.96. Pocock spends evenly and so is nearly flat, buying much more early stopping
at the price of a final look that is meaningfully harder to pass.

The budget is exhausted **exactly**, not approximately. That is the property to check.

### Does it hold?

In [ ]:
spent = sequential_error_curve(null_spec, look_counts=LOOK_COUNTS, n_sims=N_SIMS, seed=0)
bayesian = {
    k: bayes_peeking_study(null_spec, k, n_sims=N_SIMS, seed=0) for k in LOOK_COUNTS
}

fig, ax = plt.subplots()
for label, curve, colour in (
    ("naive peeking", naive, RUST),
    ("O'Brien-Fleming spending", spent, NAVY),
    ("Bayesian, P(B>A) > 0.95", bayesian, TEAL),
):
    ks = list(curve)
    rates = [curve[k].reject_rate for k in ks]
    errs = np.array([[r - curve[k].reject_rate_ci[0] for r, k in zip(rates, ks)],
                     [curve[k].reject_rate_ci[1] - r for r, k in zip(rates, ks)]])
    ax.errorbar(ks, rates, yerr=errs, fmt="o-", color=colour, lw=2, capsize=3, label=label)

ax.axhline(0.05, color="black", ls="--", label="nominal 5%")
ax.set_xlabel("number of looks before the horizon")
ax.set_ylabel("measured Type-I error")
ax.set_title("Only one of these three keeps the promise on the tin")
ax.legend()
plt.show()

That chart is the centre of the project.

- **Naive peeking** climbs steeply. At 20 looks a team believing they are wrong 1 time in 20
  is in fact wrong roughly 1 time in 4.
- **Alpha spending** is flat, because that is the thing it was constructed to be.
- **The Bayesian rule inflates too**, and at much the same rate. This is worth being precise
  about: a posterior is not invalidated by being looked at, and nothing here says otherwise.
  But a *stopping rule* built on a posterior threshold still has a frequentist error rate, and
  nothing about Bayesian machinery ever promised to control it. The two are constantly
  conflated.

### What alpha spending costs

Nothing is free. Spending the budget buys back sample size and gives up a little power.

In [ ]:
alt_spec = ExperimentSpec(
    n_per_arm=n_required, metric="binary", baseline=BASELINE, effect=EFFECT
)
rows = []
for label, spending in (("O'Brien-Fleming", "obrien-fleming"), ("Pocock", "pocock")):
    for k in (1, 5, 20):
        s = sequential_study(alt_spec, k, n_sims=N_SIMS, seed=0, spending=spending)
        rows.append((label, k, s.reject_rate, s.mean_n_per_arm, s.mean_estimate))

print(f"{'rule':<18}{'looks':>6}{'power':>9}{'mean n':>10}{'reported':>11}   (true effect 0.02)")
for label, k, power, mean_n, est in rows:
    print(f"{label:<18}{k:>6}{power:>9.4f}{mean_n:>10,.0f}{est:>11.4f}")

Twenty O'Brien–Fleming looks buy back roughly a quarter of the sample for a couple of points
of power, and keep the error rate. Pocock stops sooner still and pays considerably more power
for it — most of that at the final look, where its bar is 2.39 rather than 2.06. Choosing a
spending function is choosing where you would rather lose.

Notice the last column: the reported effect shrinks toward the truth as the boundary gets
stricter. A rule that will not stop early unless the evidence is overwhelming stops early less
often, and so selects on noise less hard. The correction that fixes the error rate improves
the estimate for free.

## 4. CUPED: buy precision with data you already have

Everything so far changes the *decision rule*. CUPED changes the *metric*, and it is the only
lever here that makes an experiment genuinely cheaper rather than differently risky.

If a pre-experiment covariate `X` predicts the outcome `Y`, then `Y - θ(X - X̄)` has the same
expectation — the effect estimate stays unbiased — but its variance falls to `Var(Y)(1 - ρ²)`.

In [ ]:
CORRS = (0.0, 0.2, 0.4, 0.6, 0.8)
continuous_spec = ExperimentSpec(
    n_per_arm=2000, metric="continuous", baseline=0.0, effect=0.05, sd=1.0
)
cont = variance_reduction_curve(continuous_spec, correlations=CORRS, n_sims=N_SIMS, seed=0)
binary = variance_reduction_curve(
    ExperimentSpec(n_per_arm=n_required, metric="binary", baseline=BASELINE, effect=EFFECT),
    correlations=CORRS,
    n_sims=N_SIMS,
    seed=0,
)

fig, ax = plt.subplots()
for label, curve, colour in (("continuous metric", cont, NAVY), ("binary metric", binary, PLUM)):
    realised = [curve[r].mean_correlation for r in CORRS]
    ax.plot(realised, [curve[r].mean_variance_reduction for r in CORRS], "o", color=colour,
            ms=9, label=f"{label} — measured")

grid = np.linspace(0, 0.85, 100)
ax.plot(grid, grid**2, "-", color=RUST, lw=2, label=r"theory: $1-(1-\rho^2) = \rho^2$")
ax.set_xlabel(r"$\it{realised}$ covariate-outcome correlation")
ax.set_ylabel("measured variance reduction")
ax.set_title("CUPED delivers exactly what the theory promises")
ax.legend()
plt.show()

print(f"{'requested':>10}{'realised':>10}{'reduction':>11}{'predicted':>11}{'power raw':>11}{'power CUPED':>13}")
for r in CORRS:
    c = binary[r]
    print(f"{r:>10.2f}{c.mean_correlation:>10.4f}{c.mean_variance_reduction:>11.4f}"
          f"{c.predicted_variance_reduction:>11.4f}{c.raw.reject_rate:>11.4f}"
          f"{c.adjusted.reject_rate:>13.4f}")

Both metrics sit on the theoretical curve — but look at the x-axis. It is the **realised**
correlation, not the requested one, and for the binary metric those are very different numbers:
asking for 0.8 yields about 0.51.

CUPED is not underperforming. It delivers exactly `1 - ρ²` against the correlation the data
actually has. The loss is in the covariate: a binary covariate is a thresholded version of
something continuous, and thresholding destroys correlation. This is why "did the user convert
last month" is a much weaker CUPED covariate than "how much did the user spend last month",
even when both look similarly predictive.

## 5. Bayesian decision rules: same arithmetic, different sentence

The last comparison. With weak priors, `P(B > A)` and the frequentist `1 - p/2` are not merely
similar — they are the same computation approached from two directions.

In [ ]:
for label, spec in (
    ("null", null_spec),
    ("2pp lift", alt_spec),
):
    print(f"--- {label} ---")
    for threshold in (0.95, 0.975, 0.99):
        a = agreement_study(
            spec, DecisionRule(prob_threshold=threshold), n_sims=N_SIMS * 2, seed=0
        )
        print(f"  P(B>A) >= {threshold:<6} agree {a.agreement_rate:.4f}   "
              f"bayes-only {a.bayes_only:>4}   freq-only {a.frequentist_only:>4}   "
              f"mean |prob gap| {a.mean_abs_probability_gap:.2e}")

The two frameworks disagree at the *conventional* thresholds and agree almost perfectly at a
**matched** one. That is the finding, and it is a threshold artefact rather than a
philosophical one:

- `P(B > A) > 0.95` is a **one-sided 5%** rule.
- A two-sided α = 0.05 ships on a positive result at **2.5%**.

So the industry-standard posterior threshold is twice as permissive in the ship direction as
the industry-standard p-value threshold, and every "Bayesian methods ship more winners"
comparison built on that pairing is measuring the thresholds, not the frameworks. Match them
at 0.975 and the disagreement rate falls by roughly a factor of 600.

The tell is in the columns: disagreements run **entirely one way**. A genuine philosophical
difference would scatter in both directions. One-directional disagreement is the signature of
a threshold mismatch, and it is a useful diagnostic to keep.

### So what is Bayesian analysis actually good for here?

Expected loss. `P(B > A) = 0.96` tells you the probability you are right; it says nothing about
what being wrong would cost. `E[max(control - treatment, 0)]` is the average amount of metric
surrendered by shipping, and a threshold on it — "ship if the expected cost of being wrong is
under 0.1 percentage points" — is a business statement rather than a statistical one. A p-value
cannot be one. That advantage survives everything above.

---

## Where this lands

| Question | Answer this repo measured |
|---|---|
| How many users do we need? | 3,841 per arm, and simulation confirms the formula delivers its 80% |
| What does peeking cost? | 5% → ~25% Type-I at 20 looks, plus a badly inflated effect estimate |
| Does alpha spending fix it? | Yes — flat at 5%, for ~2 points of power and a quarter of the sample back |
| Is CUPED worth the join? | `1 - ρ²`, exactly. At ρ = 0.8 that is worth ~2.8× the traffic |
| Does Bayes fix peeking? | No. The posterior is fine; the stopping rule built on it has the same problem |

Every one of those is reproducible from a clean clone. The CLI commands that produce the
full-precision versions are in README §6.